In [22]:
pip install json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Lectura de archivos
import kagglehub
import os
import pandas as pd
import json

In [ ]:
# Download latest version
path = kagglehub.dataset_download("datasnaek/youtube")

print("Path to dataset files:", path)

Path to dataset files: /home/jimmii/.cache/kagglehub/datasets/datasnaek/youtube/versions/24


In [ ]:
# Listar los archivos para saber el nombre exacto
archivos = os.listdir(path)
print("Archivos disponibles:", archivos)

Archivos disponibles: ['US_category_id.json', 'USvideos.csv', 'GBvideos.csv', 'GB_category_id.json', 'GBcomments.csv', 'UScomments.csv']


In [ ]:
csv_path_GBvideos= os.path.join(path, 'GBvideos.csv')
df_GBvideos = pd.read_csv(csv_path_GBvideos, on_bad_lines='skip')
csv_path_GBcomments= os.path.join(path, 'GBcomments.csv')
df_GBcomments = pd.read_csv(csv_path_GBcomments, on_bad_lines='skip')
csv_path_UScomments= os.path.join(path, 'UScomments.csv')
df_UScomments = pd.read_csv(csv_path_UScomments, on_bad_lines='skip')

/tmp/ipykernel_30337/3958609831.py:8: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_UScomments = pd.read_csv(csv_path_UScomments, on_bad_lines='skip')


In [19]:
df_GBvideos.head(5)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,jt2OHQh0HoQ,Live Apple Event - Apple September Event 2017 ...,Apple Event,28,apple events|apple event|iphone 8|iphone x|iph...,7426393,78240,13548,705,https://i.ytimg.com/vi/jt2OHQh0HoQ/default_liv...,13.09
1,AqokkXoa7uE,Holly and Phillip Meet Samantha the Sex Robot ...,This Morning,24,this morning|interview|holly willoughby|philli...,494203,2651,1309,0,https://i.ytimg.com/vi/AqokkXoa7uE/default.jpg,13.09
2,YPVcg45W0z4,My DNA Test Results! I'm WHAT?!,emmablackery,24,emmablackery|emma blackery|emma|blackery|briti...,142819,13119,151,1141,https://i.ytimg.com/vi/YPVcg45W0z4/default.jpg,13.09
3,T_PuZBdT2iM,getting into a conversation in a language you ...,ProZD,1,skit|korean|language|conversation|esl|japanese...,1580028,65729,1529,3598,https://i.ytimg.com/vi/T_PuZBdT2iM/default.jpg,13.09
4,NsjsmgmbCfc,Baby Name Challenge!,Sprinkleofglitter,26,sprinkleofglitter|sprinkle of glitter|baby gli...,40592,5019,57,490,https://i.ytimg.com/vi/NsjsmgmbCfc/default.jpg,13.09


In [32]:
df_GBvideos.describe()

,category_id,views,likes,dislikes,comment_total,date
count,7993.000000,7.993000e+03,7.993000e+03,7993.000000,7993.000000,7993.000000
mean,19.738271,1.110733e+06,3.885966e+04,1528.858877,4991.631177,16.088874
std,7.178132,3.048740e+06,1.092951e+05,8176.707788,26837.135518,7.678176
min,1.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,1.100000
25%,17.000000,1.083750e+05,2.580000e+03,71.000000,308.000000,10.100000
50%,23.000000,3.151940e+05,9.561000e+03,249.000000,1038.000000,16.100000
75%,24.000000,9.712210e+05,3.159300e+04,867.000000,3327.000000,21.100000
max,29.000000,5.896141e+07,2.289911e+06,192725.000000,813322.000000,30.090000


In [21]:
df_GBcomments.head(5)

,video_id,comment_text,likes,replies
0,jt2OHQh0HoQ,It's more accurate to call it the M+ (1000) be...,0,0
1,jt2OHQh0HoQ,To be there with a samsung phone\n😂😂😂,1,0
2,jt2OHQh0HoQ,"Thank gosh, a place I can watch it without hav...",0,0
3,jt2OHQh0HoQ,What happened to the home button on the iPhone...,0,0
4,jt2OHQh0HoQ,Power is the disease. Care is the cure. Keep...,0,0


In [ ]:
csv_path_GB_category_id = os.path.join(path, 'GB_category_id.json')
with open(csv_path_GB_category_id, 'r') as f:
    data = json.load(f)

categories = []
for category in data['items']:
    categories.append({
        'category_id': int(category['id']),
        'name': category['snippet']['title']
    })

df_GB_category_id = pd.DataFrame(categories)
df_GB_category_id.head(5)

,category_id,name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


In [29]:
# Unir Videos con Categorías (usando el ID)
# Esto añade la columna 'category_name' a videos
df_videos_con_cat = pd.merge(df_GBvideos, df_GB_category_id, on='category_id', how='left')

# Usamos 'inner' para asegurar que cada fila tenga un mensaje y estadísticas de video
df = pd.merge(df_GBcomments, df_videos_con_cat, on='video_id', how='inner', suffixes=('_com', '_vid'))

In [ ]:
# Eliminar duplicados de videos para que cada video_id sea ÚNICO
df_videos_unique = df_videos_con_cat.sort_values('trending_date').drop_duplicates('video_id', keep='last')

In [30]:
df.sample(10)

,video_id,comment_text,likes_com,replies,title,channel_title,category_id,tags,views,likes_vid,dislikes,comment_total,thumbnail_link,date,name
775705,zTTaFg2Sq9Y,Is Ryan ok with this ?,0,0,All I See Is You | Official Trailer | In Theat...,Open Road Films,24,Marc Forster|Blake Lively,2062725,4085,243,485,https://i.ytimg.com/vi/zTTaFg2Sq9Y/default.jpg,22.09,Entertainment
2901738,3iKiBx7oCP0,How come u don't use the fenty match sticks? N...,0,0,FENTY BEAUTY REVIEW + TESTING NEW GALAXY COLLE...,makeupbymichaelfinch,26,fenty beauty|honest re|fenty beauty review|mic...,204163,9381,539,1103,https://i.ytimg.com/vi/3iKiBx7oCP0/default.jpg,15.10,Howto & Style
621141,LcZ2AuvxXNA,next video....bottle flip 2...like if u agree,1,0,Nerf Bow Trick Shots | Dude Perfect,Dude Perfect,17,dude perfect|dude perfect stereotypes|dude per...,11475270,398493,8158,24861,https://i.ytimg.com/vi/LcZ2AuvxXNA/default.jpg,14.09,Sports
1917802,fs6JjOOIpXk,JAJAJA que mal traducido esta no dice nada de ...,0,0,"Diego Costa: I'm Sorry Antonio Conte, Please T...",talkSPORT,17,talksport|sport|football|funny|soccer|diego co...,47693,937,84,174,https://i.ytimg.com/vi/fs6JjOOIpXk/default.jpg,1.10,Sports
4166517,LlL_IAZ6Dr4,The only reason an attractive woman would ever...,0,0,Jessica Chastain Harvey Weinstein & Courtney L...,PopCandiesTv,24,Jessica|Chastain|Harvey|Weinstein|Courtney|Lov...,410869,215,878,302,https://i.ytimg.com/vi/LlL_IAZ6Dr4/default.jpg,19.10,Entertainment
4083231,lLOEu9p80rE,This year we're having a Star Trek themed part...,0,0,Making My Own Halloween Costume! I Tom Daley,Tom Daley,17,Tom Daley|Tom|Daley|Tom Daley TV|Diver|Diving|...,44670,2569,109,297,https://i.ytimg.com/vi/lLOEu9p80rE/default.jpg,16.10,Sports
461308,XV1EkoGWong,The only guy that i think is the most humble p...,0,0,Nick Jonas Is the New Late Late Intern,The Late Late Show with James Corden,24,James Corden|The Late Late Show|Colbert|late n...,537024,12269,102,344,https://i.ytimg.com/vi/XV1EkoGWong/default.jpg,18.09,Entertainment
312063,3VxviCvN5sU,He was born in London. He was a wonderful man.,1,0,A tribute to Sir Bruce Forsyth - Strictly Come...,BBC Strictly Come Dancing,24,Sir Bruce Forsyth|Sir Bruce Forsyth tribute|st...,472673,6807,302,536,https://i.ytimg.com/vi/3VxviCvN5sU/default.jpg,15.09,Entertainment
3957296,mywgWuDqDTQ,Barca is different from Argentina. Messi isn't...,1,17,Atletico Madrid vs Barcelona 1-1 - All Goals &...,GOLAZO TV,17,Atletico Madrid|vs|Barcelona|1-1|Goals|Highlig...,2971268,10650,1191,2485,https://i.ytimg.com/vi/mywgWuDqDTQ/default.jpg,21.10,Sports
3492972,kwANDfkOskg,How is this guy relevant,0,0,The Pengest Munch Ep. 18: Chicken Spot (Casabl...,Chicken Connoisseur,24,[none],385931,18041,756,2104,https://i.ytimg.com/vi/kwANDfkOskg/default.jpg,15.10,Entertainment


In [31]:
df.describe()

,likes_com,replies,category_id,views,likes_vid,dislikes,comment_total,date
count,4.209091e+06,4.209091e+06,4.209091e+06,4.209091e+06,4.209091e+06,4.209091e+06,4.209091e+06,4.209091e+06
mean,5.098159e+00,3.856082e-01,1.972753e+01,1.460505e+06,4.971143e+04,2.148674e+03,6.765403e+03,1.595992e+01
std,1.910924e+02,9.236461e+00,7.296641e+00,3.770806e+06,1.290415e+05,1.120876e+04,3.383152e+04,7.434210e+00
min,0.000000e+00,0.000000e+00,1.000000e+00,4.980000e+02,0.000000e+00,0.000000e+00,0.000000e+00,1.100000e+00
25%,0.000000e+00,0.000000e+00,1.700000e+01,1.660550e+05,4.452000e+03,1.160000e+02,5.150000e+02,1.110000e+01
50%,0.000000e+00,0.000000e+00,2.300000e+01,4.307290e+05,1.311100e+04,3.600000e+02,1.537000e+03,1.610000e+01
75%,0.000000e+00,0.000000e+00,2.400000e+01,1.244984e+06,4.267800e+04,1.100000e+03,4.383000e+03,2.109000e+01
max,6.063000e+04,5.210000e+02,2.900000e+01,5.896141e+07,2.289911e+06,1.927250e+05,8.133220e+05,3.009000e+01


In [ ]:
df.